# FCD — Rendre le résultat publiable : qualité des cartes (étape 1) + validation croisée 5 folds (étape 2)

Un seul notebook, piloté par la **Cell 1**. Chaque session Kaggle fait une partie du travail ; les sorties sont
**cumulatives** : il suffit d'attacher la sortie de la version précédente pour reprendre.

| Étape | Ce qui est fait | Ressource | Durée |
|---|---|---|---|
| **1 — Qualité des cartes** | AUPRC, contraste et localisation de la carte d'anomalie pour les 77 patients ; lien avec le gain hybride − baseline et avec la dépendance à la carte (test n = 27 + validation n = 10) | CPU suffit | ~10 min |
| **2a — Entraînement CV** | 5 folds sur les 50 patients d'entraînement, **fold 0 = modèles déjà entraînés** (réutilisés, rien à refaire) → 4 folds × 2 modèles à entraîner | GPU T4 | ~7,7 h, **1 fold par session** |
| **2b — Comparaison CV** | Prédictions hors-fold des 50 patients, critère pré-enregistré, test de dépendance à la carte par fold, hypothèse de mécanisme (gain ~ qualité de la carte), tableau final CV + test | CPU/GPU | ~15 min |

**Le test officiel (27 patients) n'est plus jamais utilisé pour entraîner ou choisir quoi que ce soit.** Il sert
uniquement de confirmation indépendante déjà obtenue (résultat du 04/11/2026).

**v2 (après la session du 04/11)** : les modèles sont écrits directement dans la sortie (`cv_out/nnunet_results`),
avec un point de sauvegarde toutes les 5 époques ; le temps est surveillé toutes les 30 s (et non plus seulement quand
nnU-Net écrit) ; un seul fold par session. Une session interrompue ne perd donc plus que quelques époques.

**Critères de la CV** : écrits et horodatés dans `criteres_cv.json` lors de la première session, **avant** tout
nouveau résultat de CV ; la comparaison refuse de s'exécuter sans eux.

### Mode d'emploi (8 sessions GPU + 1 finale)
1. Session 1 : `CV_MODE = 'baseline'` → étape 1 + critères CV + fold 1 baseline.
2. Sessions suivantes : attache la **sortie de la version précédente** de ce notebook, alterne ou enchaîne
   `CV_MODE = 'baseline'` / `'hybride'` ; `CV_FOLDS = 'auto'` reprend toujours le fold manquant (y compris un fold
   interrompu, depuis `checkpoint_latest.pth`).
3. Quand les 5 folds des deux modèles existent, la comparaison s'exécute automatiquement (mets `CV_MODE = None`
   pour une session finale rapide).

### Inputs à attacher
dataset prétraité (`final/`), dataset du plan (`plan_experimental.csv`), dataset des cartes (`anomaly_maps/`),
sortie du notebook **baseline** (`nnunet_baseline/`), sortie du notebook **hybride** (`nnunet_hybride/`),
sortie du **test final v2** (`test_final/`) et, à partir de la session 2, la **sortie de la version précédente de ce notebook** (`cv_out/`).

In [ ]:
# Cell 1 — Configuration de la session (seule cellule à modifier)
RUN_ANALYSE_CARTE = True   # étape 1 : qualité des cartes (rapide ; ignorée si déjà faite et attachée)
CV_MODE = 'hybride'       # 'baseline', 'hybride' ou None (pas d'entraînement dans cette session)
CV_FOLDS = 'auto'          # 'auto' = folds manquants dans l'ordre ; ou liste explicite, ex. [2]
RUN_COMPARAISON = True     # s'exécute seulement si les 5 folds des 2 modèles sont disponibles

N_FOLDS = 5
EPOCHS = 100               # identique aux modèles du fold 0 (obligatoire pour la cohérence)
ITERS_PER_EPOCH = 250
VAL_ITERS = 50
SAVE_EVERY = 5
TIME_BUDGET_H = 10.5       # arrêt propre bien avant la limite de 12 h de Kaggle
MAX_FOLDS_PER_SESSION = 1  # un fold (~8 h) par session : il tient toujours dans la limite
NNUNET_VERSION = '2.8.1'   # version vérifiée par le contrôle de reproductibilité du test final
SEED_CV = 2027             # tirage des folds 1-4 (fixé une fois pour toutes)
MIN_CLUSTER = 20           # identique à toutes les analyses précédentes
ALPHA = 0.05
N_BOOT = 10000
MAP_TOP_PCT = 99.5         # seuil de la carte pour sa propre « détection » (percentile dans le cerveau)
assert CV_MODE in (None, 'baseline', 'hybride')
print('Session :', dict(analyse=RUN_ANALYSE_CARTE, cv_mode=CV_MODE, folds=CV_FOLDS, comparaison=RUN_COMPARAISON))

In [ ]:
# Cell 2 — Installation (version figée), imports, environnement, trainer personnalisé
import os, sys, time, json, shutil, subprocess, platform, hashlib
from pathlib import Path
T0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'nnunetv2=={NNUNET_VERSION}'], check=False)

import numpy as np
import pandas as pd
import SimpleITK as sitk
import scipy.stats as ss
from scipy import ndimage
from scipy.stats import wilcoxon, binomtest, spearmanr, mannwhitneyu
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import torch, nnunetv2
from importlib.metadata import version

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working') / 'cv_out'
TMP = Path('/tmp/nnunet_cv')
for d in (WORK, TMP): d.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TRAINER, CONFIG = 'nnUNetTrainer_Kaggle', '3d_fullres'
MODELS = {'baseline': (501, 'Dataset501_FCDbase'), 'hybride': (502, 'Dataset502_FCDanom')}
TRDIR = f'{TRAINER}__nnUNetPlans__{CONFIG}'
os.environ.update(nnUNet_raw=str(TMP / 'raw'), nnUNet_preprocessed=str(TMP / 'pre'), nnUNet_results=str(TMP / 'results'),
                  nnUNet_compile='f', nnUNet_n_proc_DA=str(min(4, os.cpu_count() or 2)),
                  KAGGLE_EPOCHS=str(EPOCHS), KAGGLE_ITERS=str(ITERS_PER_EPOCH),
                  KAGGLE_VAL_ITERS=str(VAL_ITERS), KAGGLE_SAVE_EVERY=str(SAVE_EVERY))
ENV = {'python': platform.python_version(), 'nnunetv2': version('nnunetv2'), 'torch': torch.__version__,
       'numpy': np.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'}
print(json.dumps(ENV, indent=1))
if CV_MODE: assert DEVICE == 'cuda', "L'entraînement nécessite le GPU (Settings → Accelerator → GPU T4)."

TRAINER_CODE = '''import os, torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainer_Kaggle(nnUNetTrainer):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 device: torch.device = torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = int(os.environ.get('KAGGLE_EPOCHS', 100))
        self.num_iterations_per_epoch = int(os.environ.get('KAGGLE_ITERS', 250))
        self.num_val_iterations_per_epoch = int(os.environ.get('KAGGLE_VAL_ITERS', 50))
        self.save_every = int(os.environ.get('KAGGLE_SAVE_EVERY', 5))
'''
tdir = Path(nnunetv2.__file__).parent / 'training' / 'nnUNetTrainer' / 'variants' / 'kaggle'
tdir.mkdir(parents=True, exist_ok=True)
(tdir / '__init__.py').write_text('')
(tdir / 'nnUNetTrainer_Kaggle.py').write_text(TRAINER_CODE)
from nnunetv2.utilities.find_objects import recursive_find_trainer_class_by_name
assert recursive_find_trainer_class_by_name(TRAINER) is not None
print('Trainer installé.')

In [ ]:
# Cell 3 — Données communes, outils de métriques et de statistiques
def find_one(name):
    c = sorted(INPUT.glob(f'**/{name}'))
    assert c, f"{name} introuvable dans /kaggle/input : attache l'input qui le contient."
    return c[0]

def find_latest(name):
    # priorité : fichier produit dans cette session, puis inputs (sorties précédentes)
    w = sorted(WORK.glob(f'**/{name}'))
    if w: return w[0]
    c = sorted(INPUT.glob(f'**/{name}'))
    return c[0] if c else None

plan = pd.read_csv(find_one('plan_experimental.csv'))
NPZ_DIR = find_one('preproc_table.csv').parent
ROLE = plan.set_index('participant_id')['role']
def ids(role):
    return sorted(plan.loc[plan.role == role, 'participant_id'])
TRAIN_IDS = ids('fcdA_val') + ids('fcdB_val')
TEST_IDS = ids('fcdA_test') + ids('fcdB_test')
assert len(TRAIN_IDS) == 50 and len(TEST_IDS) == 27 and not set(TRAIN_IDS) & set(TEST_IDS)
c = [p for p in sorted(INPUT.glob('**/anomaly_maps')) if p.is_dir()]
assert c, "Dossier anomaly_maps introuvable."
ANOM_DIR = c[0]
NEG_SET = set(plan.loc[plan.mri_diagnosis == 'none', 'participant_id']) if 'mri_diagnosis' in plan else set()

def write_nii(arr, path, dtype=np.float32):
    img = sitk.GetImageFromArray(arr.astype(dtype)); img.SetSpacing((1.0, 1.0, 1.0))
    sitk.WriteImage(img, str(path), useCompression=True)

def channels(sid, mode, zero_map=False):
    z = np.load(NPZ_DIR / f'{sid}.npz')
    m = z['mask'] > 0
    ch = [z['flair'].astype(np.float32), z['t1'].astype(np.float32)]
    if mode == 'hybride':
        a = np.load(ANOM_DIR / f'{sid}.npz')['anom'].astype(np.float32)
        assert a.shape == m.shape
        ch.append(np.zeros_like(a) if zero_map else np.where(m, np.clip(a, -5, 10), 0).astype(np.float32))
    return ch

def run(cmd):
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, env=os.environ)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(f'Échec : {cmd[0]}')
    return r

def read_seg(path):
    return sitk.GetArrayFromImage(sitk.ReadImage(str(path))) > 0

def dice(a, b):
    return float(2 * np.logical_and(a, b).sum() / (a.sum() + b.sum() + 1e-8))

def eval_case(sid, folder):
    prob = np.load(Path(folder) / f'{sid}.npz')['probabilities'][1].astype(np.float32)
    seg = read_seg(Path(folder) / f'{sid}.nii.gz')
    z = np.load(NPZ_DIR / f'{sid}.npz'); m, roi = z['mask'] > 0, z['roi'] > 0
    assert prob.shape == m.shape == seg.shape, f'{sid} : formes {prob.shape} {seg.shape} {m.shape}'
    lab, n = ndimage.label(seg)
    comps = [k for k in range(1, n + 1) if (lab == k).sum() >= MIN_CLUSTER]
    order = sorted(comps, key=lambda k: -prob[lab == k].sum())
    hits = [bool((roi & (lab == k)).any()) for k in order]
    return {'participant_id': sid, 'role': ROLE[sid], 'protocole': ROLE[sid][3],
            'dice': dice(seg, roi), 'auprc': float(average_precision_score(roi[m], prob[m])),
            'auroc': float(roc_auc_score(roi[m], prob[m])), 'detecte': any(hits), 'hit1': any(hits[:1]),
            'n_clusters': len(order), 'n_fp_clusters': len(hits) - sum(hits),
            'vol_lesion': int(roi.sum()), 'vol_pred': int(seg.sum()), 'irm_negatif': sid in NEG_SET}

# --- statistiques ---
def boot_ci(d, n=N_BOOT, seed=0):
    v = np.asarray(d, float); rs = np.random.RandomState(seed)
    m = v[rs.randint(0, len(v), (n, len(v)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def hodges_lehmann(d):
    v = np.asarray(d, float); w = (v[:, None] + v[None, :]) / 2
    return float(np.median(w[np.triu_indices(len(v))]))

def rank_biserial(d):
    v = np.asarray(d, float); v = v[v != 0]
    if len(v) == 0: return 0.0
    r = ss.rankdata(np.abs(v))
    return float((r[v > 0].sum() - r[v < 0].sum()) / r.sum())

def paired(x, y, lower_is_better=False):
    d = (pd.Series(x).astype(float) - pd.Series(y).astype(float))
    p = float(wilcoxon(d).pvalue) if (d != 0).sum() > 0 else 1.0
    lo, hi = boot_ci(d)
    better = (d < 0) if lower_is_better else (d > 0)
    return {'moyenne': float(d.mean()), 'mediane': float(d.median()), 'hodges_lehmann': hodges_lehmann(d),
            'ic95': [lo, hi], 'p_wilcoxon': p, 'r_rang_biseriale': rank_biserial(d),
            'n_meilleur': int(better.sum()), 'n_change': int((d != 0).sum()), 'n': int(len(d))}

def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z * z / n
    c_ = (p + z * z / (2 * n)) / den
    hw = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return float(c_ - hw), float(c_ + hw)

def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); m = len(ps); adj = np.empty(m); cur = 0.0
    for i, k in enumerate(o):
        cur = max(cur, (m - i) * ps[k]); adj[k] = min(1.0, cur)
    return adj

def spearman_ci(x, y, n=5000, seed=0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    rho, p = spearmanr(x, y)
    rs = np.random.RandomState(seed); bs = []
    for _ in range(n):
        i = rs.randint(0, len(x), len(x))
        if np.ptp(x[i]) > 0 and np.ptp(y[i]) > 0: bs.append(spearmanr(x[i], y[i])[0])
    return {'rho': float(rho), 'p': float(p), 'ic95': [float(np.nanpercentile(bs, 2.5)), float(np.nanpercentile(bs, 97.5))],
            'n': int(len(x))}

def detection_stats(b, h):
    gain = [s for s in b.index if h.loc[s, 'detecte'] and not b.loc[s, 'detecte']]
    lost = [s for s in b.index if b.loc[s, 'detecte'] and not h.loc[s, 'detecte']]
    p = float(binomtest(len(gain), len(gain) + len(lost)).pvalue) if (gain or lost) else 1.0
    return {'baseline': int(b.detecte.sum()), 'hybride': int(h.detecte.sum()), 'n': len(b),
            'ic95_baseline': wilson(int(b.detecte.sum()), len(b)), 'ic95_hybride': wilson(int(h.detecte.sum()), len(h)),
            'gagnes': gain, 'perdus': lost, 'p_mcnemar_exact': p}

def jdefault(o):
    return o.item() if hasattr(o, 'item') else str(o)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
COL = {'A': '#2a6fb0', 'B': '#d9822b'}
print(f'{len(TRAIN_IDS)} patients d entraînement, {len(TEST_IDS)} de test, {len(NEG_SET)} IRM-négatifs au total')

## Étape 1 — Qualité de la carte d'anomalie et lien avec le gain du modèle hybride *(analyse exploratoire, post hoc)*

Pour chaque patient, la carte est évaluée **seule** face à la vérité terrain, dans le masque cérébral, après le même
écrêtage [−5, 10] que celui utilisé en entrée du réseau :
`map_auprc` (et `map_lift` = AUPRC / prévalence de la lésion), `map_auroc`, `map_cohen_d` (contraste lésion / cerveau sain),
`map_pct_lesion` (percentile de la médiane lésionnelle parmi les voxels sains), `map_top1` / `map_top3` (le ou les clusters
les plus forts de la carte, seuillée au percentile 99,5, touchent la lésion) et `map_n_fp` (clusters de la carte hors lésion).

In [ ]:
# Cell 4 — Étape 1a : qualité de la carte pour les 77 patients
def map_quality(sid):
    z = np.load(NPZ_DIR / f'{sid}.npz'); m, roi = z['mask'] > 0, z['roi'] > 0
    a = np.clip(np.load(ANOM_DIR / f'{sid}.npz')['anom'].astype(np.float32), -5, 10)
    v, y = a[m], roi[m]
    lin, lout = v[y], v[~y]
    thr = np.percentile(v, MAP_TOP_PCT)
    seg = (a > thr) & m
    lab, n = ndimage.label(seg)
    idx = np.arange(1, n + 1)
    size = ndimage.sum(seg, lab, idx) if n else np.array([])
    score = ndimage.sum(a, lab, idx) if n else np.array([])
    hit = ndimage.maximum(roi.astype(np.uint8), lab, idx) > 0 if n else np.array([], bool)
    keep = size >= MIN_CLUSTER
    order = np.argsort(-score[keep]); hit_k = hit[keep][order]
    prev = float(y.mean())
    ap = float(average_precision_score(y, v))
    return {'participant_id': sid, 'role': ROLE[sid], 'protocole': ROLE[sid][3],
            'split': 'test' if sid in TEST_IDS else 'entrainement', 'irm_negatif': sid in NEG_SET,
            'vol_lesion': int(roi.sum()), 'prevalence': prev,
            'map_auprc': ap, 'map_lift': ap / max(prev, 1e-12), 'map_auroc': float(roc_auc_score(y, v)),
            'map_cohen_d': float((lin.mean() - lout.mean()) / np.sqrt((lin.var() + lout.var()) / 2 + 1e-8)),
            'map_pct_lesion': float((lout < np.median(lin)).mean()),
            'map_top1': bool(hit_k[:1].any()), 'map_top3': bool(hit_k[:3].any()),
            'map_n_fp': int((~hit_k).sum())}

QC = None
prev_qc = find_latest('qualite_carte.csv')
if prev_qc is not None:
    QC = pd.read_csv(prev_qc).set_index('participant_id')
    print('Qualité des cartes reprise depuis', prev_qc)
elif RUN_ANALYSE_CARTE:
    t0 = time.time()
    QC = pd.DataFrame([map_quality(s) for s in TRAIN_IDS + TEST_IDS]).set_index('participant_id')
    print(f'Qualité des cartes calculée en {(time.time() - t0) / 60:.1f} min')
if QC is not None:
    QC.to_csv(WORK / 'qualite_carte.csv')
    qcols = ['map_auprc', 'map_lift', 'map_auroc', 'map_cohen_d', 'map_pct_lesion', 'map_top1', 'map_top3', 'map_n_fp']
    print('\nQualité de la carte par protocole et par jeu (moyennes) :')
    print(QC.groupby(['split', 'protocole'])[qcols].mean().round(3).to_string())
    print('\nComparaisons de distribution (Mann-Whitney, exploratoire) :')
    for col in ['map_auprc', 'map_cohen_d', 'map_pct_lesion']:
        pa = mannwhitneyu(QC.loc[QC.protocole == 'A', col], QC.loc[QC.protocole == 'B', col]).pvalue
        pt = mannwhitneyu(QC.loc[QC.split == 'entrainement', col], QC.loc[QC.split == 'test', col]).pvalue
        print(f'  {col:15s} A vs B p = {pa:.3f} | entraînement vs test p = {pt:.3f}')
    for col in ['map_top1']:
        t = pd.crosstab(QC.protocole, QC[col]); print(f'\n{col} par protocole :'); print(t.to_string())

In [ ]:
# Cell 5 — Étape 1b : le gain de l'hybride dépend-il de la qualité de la carte ? (test n = 27, + validation n = 10)
ANALYSE = {}
def load_test(name):
    p = sorted(INPUT.glob(f'**/test_final/test_metrics_{name}.csv'))
    return pd.read_csv(p[0]).set_index('participant_id') if p else None

tb, th, thz = (load_test(n) for n in ('baseline', 'hybride', 'hybride_sans_carte'))
prev_e1 = find_latest('etape1_resultats.json')
if prev_e1 is not None:
    ANALYSE = json.load(open(prev_e1))
    for p in ('etape1_resultats.json', 'etape1_gain_vs_qualite_carte.csv', 'fig_etape1_qualite_carte.png', 'fig_etape1_qualite_carte.pdf'):
        q = find_latest(p)
        if q is not None and not (WORK / p).exists(): shutil.copy(q, WORK / p)
    print('Étape 1b déjà faite :', prev_e1)
elif QC is None or tb is None:
    print("Étape 1b ignorée : qualité des cartes ou sortie test_final absente.")
else:
    D = pd.DataFrame({'protocole': tb.protocole, 'dice_baseline': tb.dice, 'dice_hybride': th.dice,
                      'dice_sans_carte': thz.dice, 'delta': th.dice - tb.dice, 'dependance': th.dice - thz.dice,
                      'det_b': tb.detecte, 'det_h': th.detecte, 'jeu': 'test'}).join(QC[qcols])
    # validation interne (10 patients du fold 0), si les sorties d'entraînement sont attachées
    pb = sorted(INPUT.glob('**/nnunet_baseline/val_metrics.csv')); ph = sorted(INPUT.glob('**/nnunet_hybride/val_metrics.csv'))
    pz = sorted(INPUT.glob('**/nnunet_hybride/val_metrics_sans_carte.csv'))
    if pb and ph and pz:
        vb, vh, vz = (pd.read_csv(p[0]).set_index('participant_id') for p in (pb, ph, pz))
        V = pd.DataFrame({'protocole': vb.role.str[3], 'dice_baseline': vb.dice, 'dice_hybride': vh.dice,
                          'dice_sans_carte': vz.dice, 'delta': vh.dice - vb.dice, 'dependance': vh.dice - vz.dice,
                          'det_b': vb.detecte, 'det_h': vh.detecte, 'jeu': 'validation'}).join(QC[qcols])
        DV = pd.concat([D, V])
    else:
        DV = D
    DV.to_csv(WORK / 'etape1_gain_vs_qualite_carte.csv')

    print('Corrélations de Spearman avec la qualité de la carte (exploratoire, IC95 bootstrap) :')
    for jeu, df in [('test (n=27)', D), (f'test + validation (n={len(DV)})', DV)]:
        for y in ['delta', 'dependance']:
            for x in ['map_auprc', 'map_cohen_d', 'map_pct_lesion']:
                r = spearman_ci(df[x], df[y]); ANALYSE[f'{jeu}|{y}~{x}'] = r
                print(f"  {jeu:24s} {y:10s} ~ {x:15s} rho = {r['rho']:+.2f} IC95 [{r['ic95'][0]:+.2f}, {r['ic95'][1]:+.2f}] p = {r['p']:.3f}")

    print('\nGain selon que la carte localise ou non la lésion (map_top1), test + validation :')
    for v_, g in DV.groupby('map_top1'):
        r = paired(g.dice_hybride, g.dice_baseline)
        print(f"  carte {'localise' if v_ else 'ne localise pas'} la lésion (n={len(g)}) : Dice {g.dice_baseline.mean():.3f} -> "
              f"{g.dice_hybride.mean():.3f}  diff {r['moyenne']:+.3f} IC95 [{r['ic95'][0]:+.3f}, {r['ic95'][1]:+.3f}] p = {r['p_wilcoxon']:.3f}")
        ANALYSE[f'top1={bool(v_)}'] = r
    if DV.map_top1.nunique() == 2:
        pmw = mannwhitneyu(DV.loc[DV.map_top1, 'delta'], DV.loc[~DV.map_top1, 'delta']).pvalue
        ANALYSE['top1_mannwhitney_p'] = float(pmw); print(f'  différence des gains entre les deux groupes : Mann-Whitney p = {pmw:.3f}')

    try:
        import statsmodels.formula.api as smf
        dd = DV.assign(map_auprc_z=(DV.map_auprc - DV.map_auprc.mean()) / DV.map_auprc.std())
        fit = smf.ols('delta ~ map_auprc_z + dice_baseline + C(protocole) + C(jeu)', data=dd).fit(cov_type='HC3')
        print('\nRégression (erreurs robustes HC3) : delta ~ qualité de la carte + Dice baseline + protocole + jeu')
        print(fit.summary2().tables[1].round(3).to_string())
        ANALYSE['ols'] = {k: {'coef': float(fit.params[k]), 'p': float(fit.pvalues[k])} for k in fit.params.index}
    except Exception as e:
        print('Régression non disponible :', e)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
    for i, (pr, g) in enumerate(QC.groupby('protocole')):
        for j, (sp, gg) in enumerate(g.groupby('split')):
            x = i * 3 + j
            ax[0].boxplot(gg.map_auprc, positions=[x], widths=0.6, showfliers=False)
            ax[0].scatter(np.full(len(gg), x) + np.random.RandomState(0).uniform(-0.15, 0.15, len(gg)), gg.map_auprc,
                          s=14, c=COL.get(pr, 'gray'), alpha=0.8)
    ax[0].set_xticks([0, 1, 3, 4]); ax[0].set_xticklabels(['A train', 'A test', 'B train', 'B test'])
    ax[0].set_ylabel('Anomaly-map AUPRC (map alone)'); ax[0].set_title('(a) Map quality by protocol and split')
    for k_, (y, lab) in enumerate([('delta', 'Dice gain (hybrid − baseline)'), ('dependance', 'Map reliance (Dice with − without map)')]):
        a_ = ax[k_ + 1]
        for pr in sorted(DV.protocole.unique()):
            for jeu, mk in [('test', 'o'), ('validation', '^')]:
                g = DV[(DV.protocole == pr) & (DV.jeu == jeu)]
                a_.scatter(g.map_auprc, g[y], c=COL.get(pr, 'gray'), marker=mk, s=34, edgecolor='k', lw=0.4,
                           label=f'{pr} {jeu}')
        a_.axhline(0, color='k', lw=0.8); a_.set_xlabel('Anomaly-map AUPRC'); a_.set_ylabel(lab)
        r = spearman_ci(DV.map_auprc, DV[y]); a_.set_title(f"({'b' if k_ == 0 else 'c'}) Spearman ρ = {r['rho']:+.2f}, p = {r['p']:.3f}")
    ax[1].legend(frameon=False, fontsize=8)
    plt.tight_layout()
    for ext in ('png', 'pdf'): plt.savefig(WORK / f'fig_etape1_qualite_carte.{ext}', dpi=300, bbox_inches='tight')
    plt.show()
    json.dump(ANALYSE, open(WORK / 'etape1_resultats.json', 'w'), indent=1, default=jdefault)

## Étape 2 — Validation croisée 5 folds sur les 50 patients d'entraînement

**Fold 0 = le découpage déjà utilisé** (mêmes 10 patients de validation) : les modèles baseline et hybride déjà entraînés
sont réutilisés tels quels. Les 40 autres patients sont répartis en folds 1 à 4, stratifiés par protocole, graine fixe.
Les mêmes folds servent aux deux modèles.

In [ ]:
# Cell 6 — Étape 2a : folds de la CV et critères pré-enregistrés (avant tout nouveau résultat de CV)
def load_old_split(mode):
    s = sorted(INPUT.glob(f'**/nnunet_{mode}/splits_final.json'))
    assert s, f'splits_final.json du notebook {mode} introuvable : attache sa sortie.'
    return json.load(open(s[0]))[0]
old_b, old_h = load_old_split('baseline'), load_old_split('hybride')
assert old_b == old_h, 'Les découpages des deux modèles du fold 0 diffèrent'
VAL0 = sorted(old_h['val'])
rest = [s for s in TRAIN_IDS if s not in VAL0]
FOLDS = [list(VAL0)] + [[] for _ in range(N_FOLDS - 1)]
rs = np.random.RandomState(SEED_CV); cnt = 0
for role in ['fcdA_val', 'fcdB_val']:
    L = sorted(s for s in rest if ROLE[s] == role); rs.shuffle(L)
    for s in L:
        FOLDS[1 + cnt % (N_FOLDS - 1)].append(s); cnt += 1
CV_SPLITS = [{'train': sorted(s for s in TRAIN_IDS if s not in f), 'val': sorted(f)} for f in FOLDS]
assert sorted(sum(FOLDS, [])) == sorted(TRAIN_IDS) and sum(len(f) for f in FOLDS) == 50
assert sorted(old_h['train']) == CV_SPLITS[0]['train'], 'Le fold 0 ne reproduit pas l entraînement initial'
FOLD_OF = {s: k for k, f in enumerate(FOLDS) for s in f}
prev_sp = find_latest('splits_cv.json')
if prev_sp is not None:
    assert json.load(open(prev_sp)) == CV_SPLITS, 'Folds différents de ceux des sessions précédentes'
json.dump(CV_SPLITS, open(WORK / 'splits_cv.json', 'w'), indent=1)
for k, f in enumerate(FOLDS):
    print(f'fold {k} : {len(f)} patients de validation (A {sum(ROLE[s][3] == "A" for s in f)}, '
          f'B {sum(ROLE[s][3] == "B" for s in f)})')

CRIT_CV = WORK / 'criteres_cv.json'
prev_c = sorted(INPUT.glob('**/criteres_cv.json'))
if not CRIT_CV.exists() and prev_c:
    shutil.copy(prev_c[0], CRIT_CV)                 # on garde l'original horodaté
if not CRIT_CV.exists():
    json.dump({
        'question': "Sur une validation croisée à 5 folds des 50 patients d'entraînement, la carte d'anomalie ajoutée "
                    "comme 3e canal améliore-t-elle nnU-Net, et son effet dépend-il de la qualité de la carte ?",
        'critere_principal': "Dice par patient sur les prédictions hors-fold (n = 50, chaque patient prédit par le modèle "
                             "qui ne l'a pas vu), différence appariée hybride - baseline, Wilcoxon bilatéral, alpha = 0,05",
        'hypothese_mecanisme': "H2 : la différence de Dice hybride - baseline hors-fold est positivement corrélée à l'AUPRC "
                               "de la carte seule (Spearman bilatéral, n = 50, alpha = 0,05)",
        'criteres_secondaires': ['détection (McNemar exact)', 'clusters faux positifs par patient', 'AUPRC au voxel',
                                 'variabilité entre folds (différence moyenne par fold)',
                                 'dépendance à la carte : hybride avec carte nulle, par fold'],
        'exploratoire': ['résultats par protocole', 'IRM-négatifs', 'régression delta ~ qualité de carte + Dice baseline + protocole'],
        'modeles': f'nnU-Net {CONFIG}, {EPOCHS} époques, trainer {TRAINER}, nnunetv2 {NNUNET_VERSION}, checkpoint_final, '
                   'mêmes folds pour les deux modèles ; fold 0 = modèles déjà entraînés',
        'transparence': "Le fold 0 (10 patients) a déjà été vu en validation interne et l'étape 1 (exploratoire) a été "
                        "réalisée sur le test officiel et ces 10 patients avant ces critères ; le test officiel n'est "
                        "plus utilisé. Aucun réglage ne sera modifié après les premiers résultats de CV.",
        'horodatage': time.strftime('%Y-%m-%d %H:%M:%S')}, open(CRIT_CV, 'w'), ensure_ascii=False, indent=1)
    print('Critères CV enregistrés maintenant.')
CRIT_CV_DATA = json.load(open(CRIT_CV))
print('Critères CV horodatés le', CRIT_CV_DATA['horodatage'])

In [ ]:
# Cell 7 — Étape 2b : préparation (modèles écrits DIRECTEMENT dans la sortie, reprise, plans du fold 0, prétraitement)
RES = WORK / 'nnunet_results'                     # dans /kaggle/working : sauvegardé même si la session est coupée
RES.mkdir(parents=True, exist_ok=True)
os.environ['nnUNet_results'] = str(RES)

def candidates(dname, k):
    return [p for p in sorted(INPUT.glob(f'**/nnunet_results/{dname}/{TRDIR}/fold_{k}')) if p.is_dir()]

def ckpt_epoch(p):
    try:
        return int(torch.load(p, map_location='cpu', weights_only=False).get('current_epoch', -1))
    except Exception:
        return -1

def best_candidate(dname, k):
    c = candidates(dname, k)
    fin = [p for p in c if (p / 'checkpoint_final.pth').exists()]
    if fin: return fin[0], 'final'
    lat = [p for p in c if (p / 'checkpoint_latest.pth').exists()]
    if lat: return max(lat, key=lambda p: ckpt_epoch(p / 'checkpoint_latest.pth')), 'partiel'
    return None, None

def local_fold(mode, k):
    return RES / MODELS[mode][1] / TRDIR / f'fold_{k}'

def fold_source(mode, k):                          # dossier d'un fold TERMINÉ (session courante ou inputs)
    lf = local_fold(mode, k)
    if (lf / 'checkpoint_final.pth').exists(): return lf
    src, st = best_candidate(MODELS[mode][1], k)
    return src if st == 'final' else None

if CV_MODE:
    DID, DNAME = MODELS[CV_MODE]
    RAW, PRE = TMP / 'raw', TMP / 'pre'
    DS = RAW / DNAME
    CH = ['FLAIR', 'T1'] + (['anomaly'] if CV_MODE == 'hybride' else [])
    src0, st0 = best_candidate(DNAME, 0)
    assert st0 == 'final', 'Fold 0 (modèle déjà entraîné) introuvable : attache la sortie du notebook ' + CV_MODE

    # reprise d'un fold interrompu lors d'une session précédente
    for k in range(1, N_FOLDS):
        if fold_source(CV_MODE, k) is not None: continue
        src, st = best_candidate(DNAME, k)
        if st == 'partiel' and not (local_fold(CV_MODE, k) / 'checkpoint_latest.pth').exists():
            shutil.copytree(src, local_fold(CV_MODE, k), dirs_exist_ok=True)
            print(f'fold {k} : reprise possible depuis l époque {ckpt_epoch(src / "checkpoint_latest.pth")}')
    STATUS = {k: 'final' if fold_source(CV_MODE, k) is not None else
              ('partiel' if (local_fold(CV_MODE, k) / 'checkpoint_latest.pth').exists() else 'à faire') for k in range(N_FOLDS)}
    print(f'{CV_MODE} — état des folds :', STATUS)

    if all(v == 'final' for v in STATUS.values()):
        print('Tous les folds de ce modèle sont terminés : rien à entraîner.')
    else:
        if not (DS / 'dataset.json').exists():
            for sub in ('imagesTr', 'labelsTr'): (DS / sub).mkdir(parents=True, exist_ok=True)
            t0 = time.time()
            for sid in TRAIN_IDS:
                for k, a in enumerate(channels(sid, CV_MODE)):
                    write_nii(a, DS / 'imagesTr' / f'{sid}_{k:04d}.nii.gz')
                write_nii((np.load(NPZ_DIR / f'{sid}.npz')['roi'] > 0).astype(np.uint8), DS / 'labelsTr' / f'{sid}.nii.gz', np.uint8)
            json.dump({'channel_names': {str(i): n for i, n in enumerate(CH)}, 'labels': {'background': 0, 'lesion': 1},
                       'numTraining': len(TRAIN_IDS), 'file_ending': '.nii.gz'}, open(DS / 'dataset.json', 'w'), indent=1)
            print(f'Conversion : {len(TRAIN_IDS)} cas en {(time.time() - t0) / 60:.1f} min')
        # mêmes plans que le fold 0 (patch, espacement, normalisation identiques), puis prétraitement
        PP = PRE / DNAME
        if not (PP / 'nnUNetPlans_3d_fullres').exists():
            PP.mkdir(parents=True, exist_ok=True)
            assert (src0.parent / 'plans.json').exists(), 'plans.json du fold 0 introuvable'
            shutil.copy(src0.parent / 'plans.json', PP / 'nnUNetPlans.json')
            shutil.copy(DS / 'dataset.json', PP / 'dataset.json')
            if (src0.parent / 'dataset_fingerprint.json').exists():
                shutil.copy(src0.parent / 'dataset_fingerprint.json', PP / 'dataset_fingerprint.json')
            t0 = time.time()
            run(['nnUNetv2_preprocess', '-d', str(DID), '-plans_name', 'nnUNetPlans', '-c', CONFIG,
                 '-np', os.environ['nnUNet_n_proc_DA']])
            print(f'Prétraitement : {(time.time() - t0) / 60:.1f} min')
        n_pre = len({p.name.split('.')[0].replace('_seg', '') for p in (PP / 'nnUNetPlans_3d_fullres').iterdir()})
        assert n_pre >= len(TRAIN_IDS), f'Prétraitement incomplet ({n_pre} cas)'
        json.dump(CV_SPLITS, open(PP / 'splits_final.json', 'w'), indent=1)
        c3 = json.load(open(PP / 'nnUNetPlans.json'))['configurations'][CONFIG]
        print(f"Plans du fold 0 réutilisés — patch {c3.get('patch_size')} | batch {c3.get('batch_size')} | espacement {c3.get('spacing')}")
else:
    print('CV_MODE = None : pas d entraînement dans cette session.')

In [ ]:
# Cell 8 — Étape 2c : entraînement d'UN fold (surveillance du temps toutes les 30 s, arrêt propre, reprise automatique)
TOUCHED = set()
if CV_MODE:
    todo = [k for k in (range(N_FOLDS) if CV_FOLDS == 'auto' else CV_FOLDS)
            if fold_source(CV_MODE, k) is None][:MAX_FOLDS_PER_SESSION]
    print(f'{CV_MODE} — fold à entraîner dans cette session : {todo}')
    keep = ('Epoch ', 'train_loss', 'val_loss', 'Pseudo dice', 'Epoch time', 'Yayy', 'Mean Validation Dice',
            'Training done', 'Error', 'error', 'Traceback')
    for k in todo:
        if TIME_BUDGET_H - (time.time() - T0) / 3600 < 1.0:
            print('Moins d une heure restante : fold', k, 'à la session suivante.'); break
        cmd = ['nnUNetv2_train', str(DID), CONFIG, str(k), '-tr', TRAINER, '--npz', '-device', DEVICE]
        if (local_fold(CV_MODE, k) / 'checkpoint_latest.pth').exists():
            cmd.append('--c'); print(f'Reprise du fold {k} depuis checkpoint_latest.pth')
        print('$', ' '.join(cmd), flush=True); TOUCHED.add(k)
        logp = WORK / f'train_log_{CV_MODE}_fold{k}_{int(time.time())}.txt'
        with open(logp, 'w') as lf:
            proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, text=True,
                                    env=dict(os.environ, PYTHONUNBUFFERED='1'))
        pos, stopped = 0, False
        while True:
            ret = proc.poll()
            with open(logp) as f:
                f.seek(pos); new = f.read(); pos = f.tell()
            for line in new.splitlines():
                if any(s_ in line for s_ in keep): print(line, flush=True)
            if ret is not None: break
            if time.time() - T0 > TIME_BUDGET_H * 3600:
                print('Budget de temps atteint : arrêt propre ; le fold reprendra au dernier point de sauvegarde.', flush=True)
                proc.terminate()
                try: proc.wait(timeout=180)
                except subprocess.TimeoutExpired: proc.kill(); proc.wait()
                stopped = True; break
            time.sleep(30)
        if stopped: break
        if proc.returncode != 0: raise RuntimeError(f'nnU-Net a échoué sur le fold {k} : voir {logp.name}')
        print(f'fold {k} terminé : {fold_source(CV_MODE, k) is not None}', flush=True)
    print(f'{CV_MODE} — folds terminés :', [k for k in range(N_FOLDS) if fold_source(CV_MODE, k) is not None])

In [ ]:
# Cell 9 — Étape 2d : métriques hors-fold, test de dépendance à la carte, export cumulatif
def metrics_name(mode, k): return f'cv_metrics_{mode}_fold{k}.csv'
def ablation_name(k): return f'cv_metrics_hybride_sans_carte_fold{k}.csv'

for mode in MODELS:
    for k in range(N_FOLDS):
        src = fold_source(mode, k)
        if src is None: continue
        if find_latest(metrics_name(mode, k)) is None:
            m = pd.DataFrame([eval_case(s_, src / 'validation') for s_ in FOLDS[k]]).assign(fold=k)
            m.to_csv(WORK / metrics_name(mode, k), index=False)
            print(f'{mode} fold {k} : Dice hors-fold moyen {m.dice.mean():.3f} (n={len(m)})')
        if mode == 'hybride' and DEVICE == 'cuda' and find_latest(ablation_name(k)) is None:
            inp, out = TMP / f'abl_in_{k}', TMP / f'abl_out_{k}'
            if inp.exists(): shutil.rmtree(inp)
            if out.exists(): shutil.rmtree(out)
            inp.mkdir(parents=True)
            for sid in FOLDS[k]:
                for c_, a in enumerate(channels(sid, 'hybride', zero_map=True)):
                    write_nii(a, inp / f'{sid}_{c_:04d}.nii.gz')
            run(['nnUNetv2_predict_from_modelfolder', '-i', str(inp), '-o', str(out), '-m', str(src.parent),
                 '-f', str(k), '-chk', 'checkpoint_final.pth', '--save_probabilities', '-device', DEVICE])
            m = pd.DataFrame([eval_case(s_, out) for s_ in FOLDS[k]]).assign(fold=k)
            m.to_csv(WORK / ablation_name(k), index=False)
            print(f'hybride fold {k} sans carte : Dice {m.dice.mean():.3f}')

# Export cumulatif : la sortie de CETTE version contient tout ce qu'il faut pour la suivante
ign = shutil.ignore_patterns('checkpoint_best.pth')
for mode, (did, dname) in MODELS.items():
    for k in range(1, N_FOLDS):                 # le fold 0 reste dans les sorties d'entraînement d'origine
        dst = local_fold(mode, k)
        if (dst / 'checkpoint_final.pth').exists() or (dst / 'checkpoint_latest.pth').exists():
            continue                            # déjà dans la sortie (entraîné ou repris dans cette session)
        src, _ = best_candidate(dname, k)
        if src is None: continue
        shutil.copytree(src, dst, ignore=ign, dirs_exist_ok=True)
        for f in ('plans.json', 'dataset.json', 'dataset_fingerprint.json'):
            if (src.parent / f).exists(): shutil.copy(src.parent / f, dst.parent / f)
for p in RES.glob(f'**/{TRDIR}/fold_*/checkpoint_best.pth'):
    p.unlink()                                  # inutile (on utilise checkpoint_final) : allège la sortie
for pat in ('cv_metrics_*.csv', 'qualite_carte.csv', 'criteres_cv.json', 'splits_cv.json', 'train_log_*.txt'):
    for p in sorted(INPUT.glob(f'**/cv_out/{pat}')):
        if not (WORK / p.name).exists(): shutil.copy(p, WORK / p.name)

def availability():
    return pd.DataFrame({m: ['✔' if find_latest(metrics_name(m, k)) else '·' for k in range(N_FOLDS)]
                         for m in MODELS} | {'hybride sans carte': ['✔' if find_latest(ablation_name(k)) else '·'
                                                                      for k in range(N_FOLDS)]},
                        index=[f'fold {k}' for k in range(N_FOLDS)])
print('\nÉtat de la validation croisée (métriques hors-fold disponibles) :')
print(availability().to_string())

In [ ]:
# Cell 10 — Étape 2e : comparaison finale (uniquement quand les 5 folds des deux modèles sont disponibles)
CV = {}
ready = all(find_latest(metrics_name(m, k)) for m in MODELS for k in range(N_FOLDS))
if not RUN_COMPARAISON:
    print('RUN_COMPARAISON = False.')
elif not ready:
    print('Comparaison en attente : il manque des folds (voir le tableau ci-dessus).')
else:
    assert CRIT_CV.exists(), 'criteres_cv.json absent : comparaison refusée'
    def oof(name_fn):
        df = pd.concat([pd.read_csv(find_latest(name_fn(k))) for k in range(N_FOLDS)]).set_index('participant_id')
        assert sorted(df.index) == sorted(TRAIN_IDS), 'Prédictions hors-fold incomplètes'
        return df.loc[TRAIN_IDS]
    B, H = oof(lambda k: metrics_name('baseline', k)), oof(lambda k: metrics_name('hybride', k))
    HZ = oof(ablation_name) if all(find_latest(ablation_name(k)) for k in range(N_FOLDS)) else None
    pd.concat({'baseline': B, 'hybride': H}, axis=1).to_csv(WORK / 'cv_oof_par_patient.csv')
    print(f"Critères CV horodatés le {CRIT_CV_DATA['horodatage']} | comparaison exécutée le {time.strftime('%Y-%m-%d %H:%M:%S')}")

    P = paired(H.dice, B.dice); lo, hi = P['ic95']
    print('=' * 80)
    print(f'CRITÈRE PRINCIPAL CV — Dice hors-fold, hybride − baseline (n = {P["n"]})')
    print(f'  baseline {B.dice.mean():.3f} ± {B.dice.std():.3f} | hybride {H.dice.mean():.3f} ± {H.dice.std():.3f}')
    print(f'  différence {P["moyenne"]:+.3f}  IC95 [{lo:+.3f}, {hi:+.3f}]  Wilcoxon p = {P["p_wilcoxon"]:.4f}  '
          f'(hybride meilleur : {P["n_meilleur"]}/{P["n_change"]})')
    print(f'  [complémentaire] Hodges-Lehmann {P["hodges_lehmann"]:+.3f} | r rang-bisériale {P["r_rang_biseriale"]:+.2f}')
    SIG = P['p_wilcoxon'] < ALPHA
    CONC = (f"Différence significative : l'hybride est {'MEILLEUR' if P['moyenne'] > 0 else 'MOINS BON'} que la baseline."
            if SIG else f"Pas de différence significative ; effets compatibles entre {lo:+.3f} et {hi:+.3f} de Dice.")
    print('  =>', CONC); print('=' * 80)

    FOLDTAB = pd.DataFrame({'n': B.groupby('fold').size(), 'dice_baseline': B.groupby('fold').dice.mean(),
                            'dice_hybride': H.groupby('fold').dice.mean()})
    FOLDTAB['difference'] = FOLDTAB.dice_hybride - FOLDTAB.dice_baseline
    if HZ is not None: FOLDTAB['dice_sans_carte'] = HZ.groupby('fold').dice.mean()
    FOLDTAB.to_csv(WORK / 'cv_par_fold.csv')
    print('\nVariabilité entre folds :'); print(FOLDTAB.round(3).to_string())
    print(f"  hybride meilleur dans {(FOLDTAB.difference > 0).sum()}/{N_FOLDS} folds ; écart-type des différences "
          f"{FOLDTAB.difference.std():.3f}")

    SEC = {'auprc': paired(H.auprc, B.auprc), 'n_fp_clusters': paired(H.n_fp_clusters, B.n_fp_clusters, True)}
    DET = detection_stats(B, H)
    adj = holm([SEC['auprc']['p_wilcoxon'], SEC['n_fp_clusters']['p_wilcoxon'], DET['p_mcnemar_exact']])
    SEC['auprc']['p_holm'], SEC['n_fp_clusters']['p_holm'], DET['p_holm'] = map(float, adj)
    print('\nCRITÈRES SECONDAIRES (p brut | Holm)')
    for k_, r in SEC.items():
        print(f"  {k_:14s} {r['moyenne']:+.4f} IC95 [{r['ic95'][0]:+.4f}, {r['ic95'][1]:+.4f}] p = {r['p_wilcoxon']:.3f} | {r['p_holm']:.3f}")
    print(f"  détection {DET['baseline']}/50 -> {DET['hybride']}/50 ; gagnés {DET['gagnes']} ; perdus {DET['perdus']} ; "
          f"McNemar p = {DET['p_mcnemar_exact']:.3f} | {DET['p_holm']:.3f}")
    ABL = None
    if HZ is not None:
        ABL = paired(H.dice, HZ.dice)
        print(f"  dépendance à la carte : Dice {H.dice.mean():.3f} -> {HZ.dice.mean():.3f} sans carte, "
              f"diff {ABL['moyenne']:+.3f} IC95 [{ABL['ic95'][0]:+.3f}, {ABL['ic95'][1]:+.3f}] p = {ABL['p_wilcoxon']:.4f}")

    H2 = None
    if QC is not None:
        M = pd.DataFrame({'delta': H.dice - B.dice, 'dice_baseline': B.dice, 'protocole': B.protocole}).join(QC[qcols])
        H2 = spearman_ci(M.map_auprc, M.delta)
        print(f"\nHYPOTHÈSE DE MÉCANISME H2 (pré-enregistrée) — Spearman(delta, AUPRC de la carte), n = {H2['n']} :")
        print(f"  rho = {H2['rho']:+.3f} IC95 [{H2['ic95'][0]:+.3f}, {H2['ic95'][1]:+.3f}]  p = {H2['p']:.4f}  => "
              f"{'confirmée' if (H2['p'] < ALPHA and H2['rho'] > 0) else 'non confirmée'}")
        M.to_csv(WORK / 'cv_delta_vs_qualite_carte.csv')

    print('\nPar protocole (exploratoire) :')
    PROTO = {}
    for pr in sorted(B.protocole.unique()):
        s = B.index[B.protocole == pr]; r = paired(H.loc[s, 'dice'], B.loc[s, 'dice']); PROTO[pr] = r
        print(f"  {pr} (n={len(s)}) : {B.loc[s, 'dice'].mean():.3f} -> {H.loc[s, 'dice'].mean():.3f}  diff {r['moyenne']:+.3f} "
              f"IC95 [{r['ic95'][0]:+.3f}, {r['ic95'][1]:+.3f}] p = {r['p_wilcoxon']:.3f}")
    neg = [s for s in TRAIN_IDS if s in NEG_SET]
    if neg:
        print(f'\nIRM-négatifs (descriptif, n = {len(neg)}) :')
        print(pd.DataFrame({'protocole': B.loc[neg, 'protocole'], 'dice_baseline': B.loc[neg, 'dice'],
                            'dice_hybride': H.loc[neg, 'dice']}).round(3).to_string())

    # Synthèse CV + test officiel
    TEST = None
    pt = sorted(INPUT.glob('**/test_final/resultat_principal.json'))
    if pt: TEST = json.load(open(pt[0]))
    rows = [{'analyse': 'CV 5 folds, hors-fold', 'n': 50, 'dice_baseline': B.dice.mean(), 'dice_hybride': H.dice.mean(),
             'difference': P['moyenne'], 'ic95': f"[{lo:+.3f}, {hi:+.3f}]", 'p_wilcoxon': P['p_wilcoxon'],
             'detection': f"{DET['baseline']}/50 vs {DET['hybride']}/50"}]
    if TEST:
        cp = TEST['critere_principal']
        rows.append({'analyse': 'Test officiel pré-enregistré', 'n': TEST['n'], 'dice_baseline': TEST['dice_baseline'],
                     'dice_hybride': TEST['dice_hybride'], 'difference': cp['moyenne'],
                     'ic95': f"[{cp['ic95'][0]:+.3f}, {cp['ic95'][1]:+.3f}]", 'p_wilcoxon': cp['p_wilcoxon'],
                     'detection': f"{TEST['detection']['baseline']['k']}/{TEST['n']} vs {TEST['detection']['hybride']['k']}/{TEST['n']}"})
    SYN = pd.DataFrame(rows).set_index('analyse')
    SYN.to_csv(WORK / 'synthese_cv_test.csv')
    try: (WORK / 'synthese_cv_test.tex').write_text(SYN.round(3).to_latex())
    except Exception as e: print('LaTeX non généré :', e)
    print('\nSYNTHÈSE :'); print(SYN.round(3).to_string())

    # Figures
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
    for pr in sorted(B.protocole.unique()):
        s = B.protocole == pr
        ax[0].scatter(B.dice[s], H.dice[s], c=COL.get(pr, 'gray'), s=30, edgecolor='k', lw=0.4, label=f'Protocol {pr}')
    ax[0].plot([0, 1], [0, 1], 'k--', lw=1); ax[0].set_xlabel('Dice — baseline'); ax[0].set_ylabel('Dice — hybrid')
    ax[0].set_title(f"(a) Out-of-fold Dice (n = 50), p = {P['p_wilcoxon']:.3f}"); ax[0].legend(frameon=False)
    ax[1].bar(FOLDTAB.index, FOLDTAB.difference, color='gray'); ax[1].axhline(0, color='k', lw=1)
    ax[1].axhline(P['moyenne'], color='k', ls='--', lw=1)
    if TEST: ax[1].axhline(TEST['critere_principal']['moyenne'], color='crimson', ls=':', lw=1.5, label='official test set')
    ax[1].set_xlabel('Fold'); ax[1].set_ylabel('Mean Dice difference (hybrid − baseline)')
    ax[1].set_title('(b) Fold-wise differences'); ax[1].legend(frameon=False) if TEST else None
    if H2 is not None:
        for pr in sorted(M.protocole.unique()):
            g = M[M.protocole == pr]
            ax[2].scatter(g.map_auprc, g.delta, c=COL.get(pr, 'gray'), s=30, edgecolor='k', lw=0.4, label=f'Protocol {pr}')
        ax[2].axhline(0, color='k', lw=0.8); ax[2].set_xlabel('Anomaly-map AUPRC'); ax[2].set_ylabel('Dice gain (hybrid − baseline)')
        ax[2].set_title(f"(c) H2: Spearman ρ = {H2['rho']:+.2f}, p = {H2['p']:.3f}"); ax[2].legend(frameon=False)
    plt.tight_layout()
    for ext in ('png', 'pdf'): plt.savefig(WORK / f'fig_cv_resultats.{ext}', dpi=300, bbox_inches='tight')
    plt.show()

    CV = {'criteres_cv_horodatage': CRIT_CV_DATA['horodatage'], 'environnement': ENV, 'n': 50,
          'critere_principal': P, 'conclusion': CONC, 'par_fold': FOLDTAB.reset_index().to_dict('records'),
          'secondaires': SEC, 'detection': DET, 'dependance_carte': ABL, 'H2_mecanisme': H2, 'par_protocole': PROTO,
          'test_officiel': TEST}
    json.dump(CV, open(WORK / 'cv_resultats.json', 'w'), ensure_ascii=False, indent=1, default=jdefault)

    txt = (f"In 5-fold cross-validation on the 50 training patients (criteria registered on {CRIT_CV_DATA['horodatage']}), "
           f"out-of-fold Dice was {B.dice.mean():.3f} for the baseline and {H.dice.mean():.3f} for the hybrid model "
           f"(paired difference {P['moyenne']:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}], Wilcoxon p = {P['p_wilcoxon']:.3f}); "
           f"the hybrid model was better in {(FOLDTAB.difference > 0).sum()} of {N_FOLDS} folds. "
           f"Detection was {DET['baseline']}/50 vs {DET['hybride']}/50 (exact McNemar p = {DET['p_mcnemar_exact']:.3f}). ")
    if ABL: txt += (f"Replacing the map by zeros changed the hybrid Dice by {-ABL['moyenne']:+.3f} (p = {ABL['p_wilcoxon']:.3f}). ")
    if H2: txt += (f"The association between the Dice gain and anomaly-map quality was Spearman ρ = {H2['rho']:+.2f}, 95% CI "
                   f"[{H2['ic95'][0]:+.2f}, {H2['ic95'][1]:+.2f}], p = {H2['p']:.3f}. ")
    if TEST: txt += (f"On the independent pre-registered test set (n = {TEST['n']}), the difference was "
                     f"{TEST['critere_principal']['moyenne']:+.3f} (p = {TEST['critere_principal']['p_wilcoxon']:.3f}).")
    (WORK / 'paragraphe_resultats_CV_EN.txt').write_text(txt)
    print('\n' + txt)

In [ ]:
# Cell 11 — Bilan de la session
print('Fichiers produits :')
for p in sorted(WORK.iterdir()):
    print('  ', p.name)
print(f'\nDurée de la session : {(time.time() - T0) / 3600:.2f} h')
print("Session suivante : attache la sortie de CETTE version, choisis CV_MODE dans la Cell 1, puis « Save & Run All ».")